# 06 - App interactiva con Streamlit

**Cliente:** DataTalent Solutions S.L.  
**Objetivo:** convertir el EDA del repositorio en una pagina interactiva para explorar ofertas, skills, salarios, calidad de datos y sesgos.

Este notebook documenta que queremos construir, por que lo hacemos y como ejecutar la app. La pagina Streamlit ejecutable esta en `../streamlit_app.py` para que pueda lanzarse desde terminal con `streamlit run`.

## 1. Relacion con el enunciado del proyecto

El PDF del proyecto pide algo mas que graficos: hay que interpretar los hallazgos en clave de negocio y reflexionar sobre posibles decisiones erroneas si los datos estan incompletos o sesgados.

Por eso la app se organiza en cinco bloques:

- **Mercado laboral:** volumen de ofertas por rol, ciudad y sector.
- **Skills y tecnologias:** skills pedidas en ofertas y tecnologias usadas/deseadas segun Stack Overflow.
- **Salarios y sesgos:** distribucion salarial, comparativas por grupo y auditoria de salarios ausentes.
- **Calidad de datos:** nulos, validaciones y correlaciones de variables derivadas.
- **Recomendaciones:** acciones concretas para DataTalent Solutions.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_EDA = PROJECT_ROOT / 'data' / 'eda'
DATA_CLEAN = PROJECT_ROOT / 'data' / 'clean'
APP_PATH = PROJECT_ROOT / 'streamlit_app.py'

print(f'Proyecto: {PROJECT_ROOT}')
print(f'App Streamlit: {APP_PATH}')
print(f'Existe app: {APP_PATH.exists()}')

Proyecto: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos
App Streamlit: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\streamlit_app.py
Existe app: True


## 2. Datos que alimentan la app

La app prioriza los CSV exportados por la fase EDA (`data/eda`). Si alguno no existe, usa como respaldo los archivos limpios (`data/clean`).

Los datasets principales son:

- `jobs_eda.csv` / `jobs_all_clean.csv`: ofertas, salarios, ubicacion, seniority, sector y variables derivadas.
- `job_skills_long.csv`: una fila por skill y oferta.
- `technology_rankings*.csv`: rankings de tecnologias usadas y deseadas.
- `cleaning_validation_summary*.csv`: controles de calidad generados en la limpieza.

In [2]:
def load_first_available(*paths):
    for path in paths:
        if path.exists():
            return path, pd.read_csv(path)
    return None, pd.DataFrame()

sources = {
    'ofertas': (DATA_EDA / 'jobs_eda.csv', DATA_CLEAN / 'jobs_all_clean.csv'),
    'skills': (DATA_CLEAN / 'job_skills_long.csv',),
    'ranking_tecnologias': (DATA_EDA / 'technology_rankings_eda.csv', DATA_CLEAN / 'technology_rankings.csv'),
    'validacion_limpieza': (DATA_EDA / 'cleaning_validation_summary_eda.csv', DATA_CLEAN / 'cleaning_validation_summary.csv'),
}

summary = []
for name, paths in sources.items():
    path, df = load_first_available(*paths)
    summary.append({
        'dataset': name,
        'archivo_usado': str(path.relative_to(PROJECT_ROOT)) if path else 'No encontrado',
        'filas': len(df),
        'columnas': len(df.columns),
    })

pd.DataFrame(summary)

,dataset,archivo_usado,filas,columnas
0,ofertas,data\eda\jobs_eda.csv,2211,21
1,skills,data\clean\job_skills_long.csv,4176,4
2,ranking_tecnologias,data\eda\technology_rankings_eda.csv,372,4
3,validacion_limpieza,data\eda\cleaning_validation_summary_eda.csv,11,3


## 3. Controles interactivos previstos

La barra lateral de la app permite filtrar el analisis sin modificar los datos originales:

- excluir o incluir outliers salariales;
- acotar rango salarial;
- filtrar por familia de rol, seniority, sector, modalidad, fuente y ciudad;
- ajustar el numero de elementos en rankings.

Estos filtros permiten contestar preguntas de negocio como: que skills dominan en roles de data engineering, que ciudades concentran mas ofertas o que sectores pagan mejor dentro del rango seleccionado.

In [3]:
jobs_path, jobs = load_first_available(DATA_EDA / 'jobs_eda.csv', DATA_CLEAN / 'jobs_all_clean.csv')

if jobs.empty:
    print('No se encontraron datos de ofertas.')
else:
    display(jobs.head())
    display(jobs[['salary_clean']].describe().T if 'salary_clean' in jobs.columns else 'No hay columna salary_clean')

,job_id,job_title,company,location,salary,job_type,post_date,link,skills,industry,...,source_dataset,salary_clean,location_clean,city_clean,is_remote,salary_clean_outlier,job_family,work_modality,post_date_parsed,post_month
0,job_00001,data scientist,company_003,"Grapevine, TX . Hybrid","€100,472 - €200,938",NaN,17 days ago,NaN,"['spark', 'r', 'python', 'scala', 'machine lea...",Retail,...,df_jobs,150705.0,"Grapevine, TX . Hybrid",Grapevine,False,False,data_science_ai,hybrid,NaN,NaN
1,job_00002,data scientist,company_005,"Fort Worth, TX . Hybrid","€118,733",NaN,15 days ago,NaN,"['spark', 'r', 'python', 'sql', 'machine learn...",Manufacturing,...,df_jobs,118733.0,"Fort Worth, TX . Hybrid",Fort Worth,False,False,data_science_ai,hybrid,NaN,NaN
2,job_00003,data scientist,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...","€94,987 - €159,559",NaN,a month ago,NaN,"['aws', 'git', 'python', 'docker', 'sql', 'mac...",Technology,...,df_jobs,127273.0,"Austin, TX . Toronto, Ontario, Canada . Kirkla...",Austin,False,False,data_science_ai,unknown,NaN,NaN
3,job_00004,data scientist,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...","€112,797 - €194,402",NaN,8 days ago,NaN,"['sql', 'r', 'python']",Technology,...,df_jobs,153599.5,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",Chicago,False,False,data_science_ai,hybrid,NaN,NaN
4,job_00005,data scientist,company_009,On-site,"€114,172 - €228,337",NaN,3 days ago,NaN,[],Finance,...,df_jobs,171254.5,On-site,On-site,False,False,data_science_ai,onsite,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
salary_clean,1105.0,113868.308145,126923.914433,17.0,39984.5,116457.0,165588.5,2739979.0


## 4. Decisiones de diseno analitico

La app no intenta sustituir al notebook EDA completo. Su funcion es convertir los resultados en una herramienta de exploracion para una audiencia mixta: tecnica y no tecnica.

Decisiones importantes:

- usamos **mediana salarial** junto a la media porque los salarios tienen dispersion y outliers;
- mostramos **porcentaje de salario informado** para no ocultar problemas de cobertura;
- separamos **skills de ofertas** y **tecnologias usadas/deseadas** porque responden a preguntas distintas;
- no inferimos sesgo de genero si el dataset de ofertas no contiene genero;
- cada visualizacion incluye una lectura de negocio breve.

## 5. Ejecutar la pagina Streamlit

Antes de lanzar la app, instala dependencias si hace falta:

```bash
pip install -r requirements.txt
```

Despues, desde la raiz del repositorio:

```bash
streamlit run streamlit_app.py
```

Tambien puedes ejecutar la siguiente celda desde Jupyter. Si Streamlit esta instalado, abrira un servidor local y mostrara la URL en la salida.

In [4]:
import subprocess
import sys

if not APP_PATH.exists():
    raise FileNotFoundError(f'No existe {APP_PATH}')

command = [sys.executable, '-m', 'streamlit', 'run', str(APP_PATH)]
print('Comando para lanzar la app:')
print(' '.join(command))

# Descomenta estas lineas si quieres lanzarla desde el notebook.
# process = subprocess.Popen(command, cwd=PROJECT_ROOT)
# print('Streamlit arrancando. Revisa la URL local en la salida del servidor.')

Comando para lanzar la app:
c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\venv\Scripts\python.exe -m streamlit run c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\streamlit_app.py


## 6. Checklist del PDF cubierto por la app

- **Exploracion:** KPIs, volumen de ofertas, tipos de variables y resumen de datasets.
- **Limpieza:** controles de nulos, outliers salariales y validaciones de limpieza.
- **Analisis estadistico:** medias, medianas, distribuciones, comparativas por grupo y correlaciones.
- **Sesgos:** ausencia de salario por grupo, outliers, representatividad por fuente y limitacion explicita sobre genero.
- **Visualizacion:** barras, dispersion, histograma con boxplot marginal, boxplot, heatmap y tablas interactivas.
- **Interpretacion:** cada bloque de la app incluye lectura de negocio y recomendaciones.